In [ ]:
GPT_CONFIG = {
     "vocab_size": 50257,    # Vocabulary size
    "context_length": 256, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": True       # Query-Key-Value bias
}

In [ ]:
# #DATALOADER to get instance of dataset
# def create_dataloader_v1(txt, batch_size=8, max_length=256,
#                          stride=128, shuffle=True, drop_last=True,
#                          num_workers=0):

#     # Initialize the tokenizer
#     tokenizer = tiktoken.get_encoding("gpt2")

#     # Create dataset
#     dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

#     # Create dataloader
#     dataloader = DataLoader(
#         dataset,
#         batch_size=batch_size,
#         shuffle=shuffle,
#         drop_last=drop_last,
#         num_workers=num_workers
#     )

#     return dataloader
def create_dataloader_v1(txt, batch_size=12, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True
    )
    return dataloader

In [ ]:
# #TO CREATE DATASET WITH INPUT AND OUTPUT PAIR DICTS
# from torch.utils.data import Dataset, DataLoader


# class GPTDatasetV1(Dataset):
#     def __init__(self, txt, tokenizer, max_length, stride):
#         self.input_ids = []
#         self.target_ids = []

#         # Tokenize the entire text
#         token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

#         # Use a sliding window to chunk the book into overlapping sequences of max_length
#         for i in range(0, len(token_ids) - max_length, stride):
#             input_chunk = token_ids[i:i + max_length]
#             target_chunk = token_ids[i + 1: i + max_length + 1]
#             self.input_ids.append(torch.tensor(input_chunk))
#             self.target_ids.append(torch.tensor(target_chunk))

#     def __len__(self):
#         return len(self.input_ids)

#     def __getitem__(self, idx):
#         return self.input_ids[idx], self.target_ids[idx]
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        self.max_length = max_length
        self.stride = stride

    def __len__(self):
        return (len(self.token_ids) - self.max_length) // self.stride

    def __getitem__(self, idx):
        start = idx * self.stride
        input_chunk  = self.token_ids[start : start + self.max_length]
        target_chunk = self.token_ids[start + 1 : start + self.max_length + 1]
        return torch.tensor(input_chunk), torch.tensor(target_chunk)

In [ ]:
import torch
import torch.nn as nn

# class GGPT(nn.Module):
#     def __init__(self, cfg):
#         super().__init__()
#         self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
#         self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
#         self.drop_emb = nn.Dropout(cfg["drop_rate"])


#         self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

#         self.final_norm = LayerNorm(cfg["emb_dim"])
#         self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias= False)

#     def forward(self, x):
#         batch_size, seq_lenth = x.shape
#         tok_emb = self.tok_emb(x)
#         pos_emb = self.pos_emb(torch.arange(seq_lenth, device=x.device))
#         x = tok_emb + pos_emb
#         x = self.drop_emb(x)
#         x = self.trf_blocks(x)
#         x = self.final_norm(x)
#         logits = self.out_head(x)
#         return logits

# class  TransformerBlock(nn.Module):
#     def __init__(self, cfg):
#         super().__init__()

#     def forward(self, x):
#         return x

## Layer Norm
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [ ]:
import tiktoken
import torch
tokenizer = tiktoken.get_encoding("gpt2")
batch= []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [ ]:
#CREATE AN INSTANCE OF DUMMYGPTMODEL
# torch.manual_seed(123)
# model = GGPT(GPT_CONFIG)
# logits=model(batch)
# print(logits.shape,logits)

In [ ]:
class GELU(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self,x):
    return  0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


In [ ]:
class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
        nn.GELU(),
        nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
    )
  def forward(self, x):
    return self.layers(x)

In [ ]:
#CAN IGNORE
import torch.nn as nn
class  DeepNeuralNetwork(nn.Module):
  def __init__(self,layer_sizes, use_shortcut):
    super().__init__()
    self.use_shortcut =use_shortcut
    self.layers = nn.ModuleList(
      [
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU())
      ]
      )

  def forward(self, x):
        for layer in self.layers:
            # Compute the output of the current layer
            layer_output = layer(x)
            # Check if shortcut can be applied
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output
        return x


In [ ]:
layer_sizes = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1., 0., -1.]])
torch.manual_seed(123) # specify random seed for the initial weights for reproducibility
model_without_shortcut = DeepNeuralNetwork(
layer_sizes, use_shortcut=False
)


In [ ]:
#Can ignore
def print_gradients(model, x):
    # Forward pass
    output = model(x)
    target = torch.tensor([[0.]])

    # Calculate loss based on how close the target
    # and output are
    loss = nn.MSELoss()
    loss = loss(output, target)

    # Backward pass to calculate the gradients
    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            # Print the mean absolute gradient of the weights
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

In [ ]:
# class MultiHeadAttention(nn.Module):
#     def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
#         super().__init__()
#         assert (d_out % num_heads == 0), \
#             "d_out must be divisible by num_heads"

#         self.d_out = d_out
#         self.num_heads = num_heads
#         self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

#         self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
#         self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
#         self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
#         self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
#         self.dropout = nn.Dropout(dropout)
#         self.register_buffer(
#             "mask",
#             torch.triu(torch.ones(context_length, context_length),
#                        diagonal=1)
#         )

#     def forward(self, x):
#         b, num_tokens, d_in = x.shape

#         keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
#         queries = self.W_query(x)
#         values = self.W_value(x)

#         # We implicitly split the matrix by adding a `num_heads` dimension
#         # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
#         keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
#         values = values.view(b, num_tokens, self.num_heads, self.head_dim)
#         queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

#         # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
#         keys = keys.transpose(1, 2)
#         queries = queries.transpose(1, 2)
#         values = values.transpose(1, 2)

#         # Compute scaled dot-product attention (aka self-attention) with a causal mask
#         attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

#         # Original mask truncated to the number of tokens and converted to boolean
#         mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

#         # Use the mask to fill attention scores
#         attn_scores.masked_fill_(mask_bool, -torch.inf)

#         attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
#         attn_weights = self.dropout(attn_weights)

#         # Shape: (b, num_tokens, num_heads, head_dim)
#         context_vec = (attn_weights @ values).transpose(1, 2)

#         # Combine heads, where self.d_out = self.num_heads * self.head_dim
#         context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
#         context_vec = self.out_proj(context_vec) # optional projection

#         return context_vec
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.dropout_p = dropout

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys    = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        context_vec = torch.nn.functional.scaled_dot_product_attention(
            queries, keys, values,
            attn_mask=None,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=True
        )

        context_vec = context_vec.transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

In [ ]:
# class TransformerBlock(nn.Module):
#     def __init__(self, cfg):
#         super().__init__()
#         self.att=MultiHeadAttention(d_in=cfg["emb_dim"],
#             d_out=cfg["emb_dim"],
#             context_length=cfg["context_length"],
#             num_heads=cfg["n_heads"],
#             dropout=cfg["drop_rate"],
#             qkv_bias=cfg["qkv_bias"])
#         self.ff = FeedForward(cfg)
#         self.norm1 = LayerNorm(cfg["emb_dim"])
#         self.norm2 = LayerNorm(cfg["emb_dim"])
#         self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

#     def forward(self, x):
#         shortcut = x
#         x = self.norm1(x)
#         x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
#         x = self.drop_shortcut(x)
#         x = x + shortcut  # Add the original input back

#         # Shortcut connection for feed forward block
#         shortcut = x
#         x = self.norm2(x)
#         x = self.ff(x)
#         x = self.drop_shortcut(x)
#         x = x + shortcut  # Add the original input back

#         return x

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])   # replaced
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])   # replaced
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = nn.LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )
        self.out_head.weight = self.tok_emb.weight

        self.apply(self._init_weights)  # ← THIS LINE IS MISSING

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):

            torch.nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

         #C1
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG)
out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[-0.2936, -0.7855,  0.8593,  ...,  0.1484,  1.1932, -0.0701],
         [-0.0217, -0.4436,  1.0877,  ...,  0.4162,  0.4408,  0.6309],
         [-0.2403, -1.0795,  0.5407,  ...,  0.2836,  0.2649,  0.4205],
         [-0.1893, -0.7389,  0.7710,  ...,  0.1313,  0.5406, -0.4238]],

        [[ 0.7259,  0.0305,  0.0106,  ...,  1.0016,  0.9903, -0.5524],
         [-0.2703, -0.9626,  1.6328,  ...,  0.7478,  0.3772, -0.8558],
         [-0.0278, -0.1797,  0.6162,  ...,  0.9560,  0.9884, -0.6735],
         [ 0.1411, -0.1166,  0.2767,  ...,  1.0288,  0.2994, -0.6324]]],
       grad_fn=<UnsafeViewBackward0>)


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 123,849,984


In [ ]:
total_params_gpt2 = total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")

Number of trainable parameters considering weight tying: 85,252,608


In [ ]:
total_size_bytes = total_params * 4 #A
total_size_mb = total_size_bytes / (1024 * 1024) #B
print(f"Total size of the model: {total_size_mb:.2f} MB")

Total size of the model: 472.45 MB


In [ ]:
# def generate_text_simple(model, idx, max_new_tokens, context_size):
#     # idx is (batch, n_tokens) array of indices in the current context
#     for _ in range(max_new_tokens):

#         # Crop current context if it exceeds the supported context size
#         # E.g., if LLM supports only 5 tokens, and the context size is 10
#         # then only the last 5 tokens are used as context
#         idx_cond = idx[:, -context_size:]

#         # Get the predictions
#         with torch.no_grad():
#             logits = model(idx_cond)

#         # Focus only on the last time step
#         # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
#         logits = logits[:, -1, :]

#         # Apply softmax to get probabilities
#         probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

#         # Get the idx of the vocab entry with the highest probability value
#         idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

#         # Append sampled index to the running sequence
#         idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

#     return idx

In [ ]:
# FOR DECODING AND ENCODIND TEXT for verifying

import tiktoken

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

# start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

# token_ids = generate_text_simple(
#     model=model,
#     idx=text_to_token_ids(start_context, tokenizer),
#     max_new_tokens=10,
#     context_size=GPT_CONFIG["context_length"]
# )

# print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-v1"
)
train_data = "\n".join(dataset["train"]["text"])
train_data = train_data[:int(len(train_data) * 0.2)]
val_data = "\n".join(dataset["validation"]["text"])

print(len(train_data))
print(len(val_data))

107777886
1141839


In [ ]:
total_characters = len(train_data)
total_tokens = len(tokenizer.encode(train_data))

print("Characters:", total_characters)
print("Tokens:", total_tokens)



Characters: 107777886
Tokens: 23692678


In [ ]:

# split_idx = int(train_ratio * len(text_data))
# train_data = text_data[:split_idx]
# val_data = text_data[split_idx:]

# train_data = dataset["train"]["text"]
# val_data = dataset["validation"]["text"]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=12,
    max_length=GPT_CONFIG["context_length"],
    stride=GPT_CONFIG["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=2
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=12,
    max_length=GPT_CONFIG["context_length"],
    stride=GPT_CONFIG["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=2
)



In [ ]:
# Sanity check

# if total_tokens * (train_ratio) < GPT_CONFIG["context_length"]:
#     print("Not enough tokens for the training loader. "
#           "Try to lower the `GPT_CONFIG_124M['context_length']` or "
#           "increase the `training_ratio`")

# if total_tokens * (1-train_ratio) < GPT_CONFIG["context_length"]:
#     print("Not enough tokens for the validation loader. "
#           "Try to lower the `GPT_CONFIG_124M['context_length']` or "
#           "decrease the `training_ratio`")
print("Total train batches:", len(train_loader))

Total train batches: 7712


In [ ]:
# print("Train loader:")
# for x, y in train_loader:
#     print(x.shape, y.shape)

# print("\nValidation loader:")
# for x, y in val_loader:
#     print(x.shape, y.shape)

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Note:
# Uncommenting the following lines will allow the code to run on Apple Silicon chips, if applicable,
# which is approximately 2x faster than on an Apple CPU (as measured on an M3 MacBook Air).
# However, the resulting loss values may be slightly different.

# if torch.cuda.is_available():
#    device = torch.device("cuda")
# elif torch.backends.mps.is_available():
#    device = torch.device("mps")
# else:
#    device = torch.device("cpu")

# print(f"Using {device} device.")


model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes


torch.manual_seed(123) # For reproducibility due to the shuffling in the data loader

with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device,num_batches=100)
    val_loss = calc_loss_loader(val_loader, model, device,num_batches=100)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 11.032048530578614
Validation loss: 11.037589485262647


In [ ]:
# def train_model_simple(model, train_loader, val_loader, optimizer,scheduler, device, num_epochs,
#                        eval_freq, eval_iter, start_context, tokenizer):
#     # Initialize lists to track losses and tokens seen
#     train_losses, val_losses, track_tokens_seen = [], [], []
#     tokens_seen, global_step = 0, -1

#     # Main training loop
#     for epoch in range(num_epochs):
#         model.train()  # Set model to training mode

#         for input_batch, target_batch in train_loader:
#             optimizer.zero_grad() # Reset loss gradients from previous batch iteration
#             loss = calc_loss_batch(input_batch, target_batch, model, device)
#             torch.nn.utils.clip_grad_norm_(
#     model.parameters(),
#     max_norm=1.0
# )
#             loss.backward() # Calculate loss gradients
#             optimizer.step() # Update model weights using loss gradients
#             scheduler.step()
#             tokens_seen += input_batch.numel() # Returns the total number of elements (or tokens) in the input_batch.
#             global_step += 1

#             # Optional evaluation step
#             if global_step % eval_freq == 0:
#                 train_loss, val_loss = evaluate_model(
#                     model, train_loader, val_loader, device, eval_iter)
#                 train_losses.append(train_loss)
#                 val_losses.append(val_loss)
#                 track_tokens_seen.append(tokens_seen)
#                 print(f"Ep {epoch+1} (Step {global_step:06d}): "
#                       f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")


#         # # generate_and_print_sample(
#         #     model, tokenizer, device, start_context
#         # )

#     return train_losses, val_losses, track_tokens_seen
from torch.cuda.amp import autocast, GradScaler

def train_model_simple(model, train_loader, val_loader, optimizer, scheduler,
                       device, num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    scaler = GradScaler()

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()

            with autocast():
                loss = calc_loss_batch(input_batch, target_batch, model, device)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        out = generate(
        model=model,
        idx=text_to_token_ids("Every effort moves you", tokenizer),
        max_new_tokens=50,
        context_size=GPT_CONFIG["context_length"],
        temperature=0.8,
        top_k=50,
        eos_id=None
       )
        generated_text = tokenizer.decode(out[0].tolist())


        print(generated_text)
        print("-" * 50)

    return train_losses, val_losses, track_tokens_seen

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [ ]:
# def generate_and_print_sample(model, tokenizer, device, start_context):
#     model.eval()
#     context_size = model.pos_emb.weight.shape[0]
#     encoded = text_to_token_ids(start_context, tokenizer).to(device)
#     with torch.no_grad():
#         token_ids = generate_text_simple(
#             model=model, idx=encoded,
#             max_new_tokens=50, context_size=context_size
#         )
#     decoded_text = token_ids_to_text(token_ids, tokenizer)
#     print(decoded_text.replace("\n", " "))  # Compact print format
#     model.train()

def generate(model,idx,max_new_tokens, context_size,temperature=0.8,top_k=50,eos_id=None):

    model.eval()

    for _ in range(max_new_tokens):

        # Keep only the last context_size tokens
        idx_cond = idx[:, -context_size:]

        # Forward pass
        with torch.no_grad():
            logits = model(idx_cond)

        # Take logits of the last token
        logits = logits[:, -1, :]

        # Top-k sampling
        if top_k is not None:

            top_logits, _ = torch.topk(
                logits,
                top_k
            )

            min_val = top_logits[:, -1].unsqueeze(-1)

            logits = torch.where(
                logits < min_val,

                torch.full_like(
                    logits,
                    float("-inf")
                ),

                logits
            )

        # Temperature sampling
        if temperature > 0:

            logits = logits / temperature

            probs = torch.softmax(
                logits,
                dim=-1
            )

            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

        # Greedy decoding
        else:

            idx_next = torch.argmax(
                logits,
                dim=-1,
                keepdim=True
            )

        # Stop if EOS token generated
        if eos_id is not None:

            if (idx_next == eos_id).all():
                break

        # Append generated token
        idx = torch.cat(
            (idx, idx_next),
            dim=1
        )

    return idx

In [ ]:
# import time
# start_time = time.time()
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# torch.manual_seed(123)
# model = GPTModel(GPT_CONFIG)
# model.to(device)
# optimizer = torch.optim.AdamW( model.parameters(),
#     lr=3e-4,
#     weight_decay=0.1)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer,
#     T_max=10
# )

# num_epochs = 4
# train_losses, val_losses, tokens_seen = train_model_simple(
#     model, train_loader, val_loader, optimizer,scheduler, device,
#     num_epochs=num_epochs, eval_freq=1000, eval_iter=10,
#     start_context="Every effort moves you", tokenizer=tokenizer
# )

# # Note:
# # Uncomment the following code to show the execution time
# end_time = time.time()
# execution_time_minutes = (end_time - start_time) / 60
# print(f"Training completed in {execution_time_minutes:.2f} minutes.")

import time
start_time = time.time()

device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using {device} device.")
num_epochs = 4
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

total_steps = num_epochs * len(train_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)  # fixed

num_epochs = 4
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, scheduler, device,
    num_epochs=num_epochs, eval_freq=1000, eval_iter=10,
    start_context="Every effort moves you", tokenizer=tokenizer
)



end_time = time.time()
print(f"Training completed in {(end_time - start_time) / 60:.2f} minutes.")

Using cuda device.


/tmp/ipykernel_1985/2605514256.py:46: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_1985/2605514256.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Ep 1 (Step 000000): Train loss 10.203, Val loss 10.180
Ep 1 (Step 001000): Train loss 5.748, Val loss 5.647


In [ ]:

# torch.manual_seed(123)

# token_ids = generate(
#     model=model,
#     idx=text_to_token_ids("Every effort moves you", tokenizer),
#     max_new_tokens=15,
#     context_size=GPT_CONFIG["context_length"],
#     top_k=25,
#     temperature=1.4
# )

# print("Output text:\n", token_ids_to_text(token_ids, tokenizer))